In [ ]:
import pandas as pd
import xarray as xr
from io import StringIO
import glob
import numpy as np
import matplotlib.pyplot as plt
import sys
from datetime import datetime, timedelta
sys.path.append('..')
from functions import HelperFunctions, CalcFunctions
from pathlib import Path

In [ ]:
def read_shadoz(filename):
    metadata = {}

    with open(filename, "r") as f:
        lines = f.readlines()

    # Find the line containing the column names
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith("Time"):
            header_idx = i
            break

        if ":" in line:
            key, value = line.split(":", 1)
            metadata[key.strip()] = value.strip()

    if header_idx is None:
        raise ValueError("Could not find data header.")

    # Read the data table
    data = "".join(lines[header_idx:])

    df = pd.read_csv(
        StringIO(data),
        sep=r"\s+",
        skiprows=[1],        # Skip the units row
        engine="python",
    )

    # Replace missing values
    missing = float(metadata.get("Missing or bad values", 9000))
    df = df.replace(missing, np.nan)

    # Convert to xarray
    ds = xr.Dataset.from_dataframe(df)

    # Store metadata
    ds.attrs.update(metadata)

    if "Launch Date" in metadata:
        launch_day = pd.to_datetime(
            metadata["Launch Date"],
            format="%Y%m%d"
        )

        ds = ds.expand_dims(
            day=[launch_day]
        )

    return ds


r_earth =       6371229.
rh2o =          461.5250
rd =            287.0597
g_earth =       9.80665
cp =            1004.7090
Omega =         7.29e-5

def calc_lrt_id(dtdz, z, dz, gamma_lrt=-2):

    dtdz_prev = np.concatenate([[-np.inf], dtdz[:-1]])

    crossings = np.where(
        (dtdz > gamma_lrt) &
        (dtdz_prev <= gamma_lrt)
    )[0]

    for idx in crossings:

        if z[idx] <= 5:
            continue

        z0 = z[idx]

        end = np.searchsorted(
            z,
            z0 + 2.0,
            side="left"
        )

        if end <= idx + 1:
            continue

        z_window = z[idx:end]
        dtdz_window = dtdz[idx:end]
        dz_window = dz[idx:end]

        if len(z_window) < 2:
            continue

        if np.any(np.diff(z_window) <= 0):
            continue

        weighted_sum = np.cumsum(
            dtdz_window * dz_window
        )

        distance = np.cumsum(dz_window)

        averages = weighted_sum / distance

        if np.all(averages >= gamma_lrt):
            return idx

    return np.nan


        
def calc_dtdz(t, z):
    return np.gradient(t, z, axis=0)

def calc_lrt(ds, gamma_lrt=-2):
    z = ds.GeopAlt
    T = ds.Temp

    dz = z.diff("height")
    dT = T.diff("height")

    dtdz = dT / dz

    z_mid = (
        (z.shift(height=-1) + z)
        .isel(height=slice(0, -1))
        / 2
    )

    z_mid = z_mid.assign_coords(height=dtdz.height)

    lrt_id = xr.apply_ufunc(
        calc_lrt_id,
        dtdz,
        z_mid,
        dz,
        input_core_dims=[["height"], ["height"], ["height"]],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    ).compute()

    valid = lrt_id.notnull()
    lrt_id_idx = lrt_id.fillna(0).astype(int)

    z_before = z_mid.isel(height=lrt_id_idx - 1)
    z_after = z_mid.isel(height=lrt_id_idx)

    dtdz_before = dtdz.isel(height=lrt_id_idx - 1)
    dtdz_after = dtdz.isel(height=lrt_id_idx)

    z_lrt = (
        z_before
        + (gamma_lrt - dtdz_before)
        * (z_after - z_before)
        / (dtdz_after - dtdz_before)
    )

    z_lrt = z_lrt.where(valid)

    trop_rel_z = z - z_lrt

    ds["z_lrt"] = z_lrt
    ds["trop_rel_z"] = trop_rel_z

    return ds


def RH_to_ppmv(ds):
    es = 6.112 * np.exp(
        (17.67 * ds.Temp) / (ds.Temp + 243.5)
    )

    e = (ds.RH / 100) * es

    ppmv = (e / (ds.Press - e)) * 1e6

    return ppmv

In [ ]:
path = '/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/Data/'
filenames_kuala = glob.glob(path + 'KualaLumpur/kuala*_SHADOZV06.dat')
filenames_nairobi = glob.glob(path + 'Nairobi/nairobi*_SHADOZV06.dat')
#filenames_quito = glob.glob(path + '/*.dat')
filenames_sancrist = glob.glob(path + 'SanCristobal/sancrist*.dat')

data_sets = []
for filename in filenames_kuala:
    data = read_shadoz(filename)
    data_sets.append(data)

data_kuala = xr.concat(data_sets, dim = 'day')

data_sets = []
for filename in filenames_nairobi:
    data = read_shadoz(filename)
    data_sets.append(data)

data_nairobi = xr.concat(data_sets, dim = 'day')

data_sets = []
for filename in filenames_sancrist:
    data = read_shadoz(filename)
    data_sets.append(data)

data_sancrist = xr.concat(data_sets, dim = 'day')

In [ ]:
fields_list = []

years = np.arange(2019,2024+1)

for year in years:

    path = f"/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/{year}/"

    months_array = np.arange(1, 13)
    months = [f"{m:02d}" for m in months_array]

    filenames_fields = [
        path + f"MeanField_{year}{m}.zarr"
        for m in months
    ]

    ds = xr.open_mfdataset(
        filenames_fields,
        engine="zarr",
        combine="nested",
        concat_dim="time"
    )

    ds = ds.rename(time = 'month')
    ds = ds.assign_coords(month = months_array)
    #ds = ds.rename(time = 'month').assign_coords(month = months_array)

    fields_list.append(ds)

fields = xr.concat(
    fields_list,
    dim="year",
)

lat_kuala = data_kuala.GPS_Lat.mean()
lon_kuala = data_kuala.GPS_Lon.mean()
lat_nairobi = data_nairobi.GPS_Lat.mean()
lon_nairobi = data_nairobi.GPS_Lon.mean()
lat_sancrist = data_sancrist.GPS_Lat.mean()
lon_sancrist = data_sancrist.GPS_Lon.mean()

In [ ]:
grid_path = '/scratch/era5/grid_era5.nc'
data_path = '/scratch/era5/'
ml_vars = {'o3', 't', 'q', 'u', 'v'}
B_vars = {'sp', 'z'}

()

def load_era5_at_location(
    ds,
    lat,
    lon,
    data_path,
    grid,
    ml_vars,
    B_vars,
):
    filelist_ml, filelist_B = zip(
        *[create_era5_filename(data_path, d) for d in ds["day"].data]
    )

    print(f"Opening {len(filelist_ml)} ML files and {len(filelist_B)} B files...")

    ds_ml = xr.open_mfdataset(
        filelist_ml,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )[list(ml_vars)]

    ds_B = xr.open_mfdataset(
        filelist_B,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )[list(B_vars)]

    print("Files opened")

    # Select location BEFORE merging
    ds_ml = ds_ml.sel(
        latitude=lat,
        longitude=lon,
        method="nearest",
    )

    ds_B = ds_B.sel(
        latitude=lat,
        longitude=lon,
        method="nearest",
    )

    print("Location selected")

    field = xr.merge([ds_ml, ds_B])

    field = CalcFunctions.calc_vertical(field, grid)

    field = CalcFunctions.calc_lrt(field).chunk("auto")

    return field

grid = HelperFunctions.load_grid(grid_path)

field_kuala = load_era5_at_location(
    ds=ds_kuala,
    lat=lat_kuala,
    lon=lon_kuala,
    data_path=data_path,
    grid=grid,
    ml_vars=ml_vars,
    B_vars=B_vars,
)


field_nairobi = load_era5_at_location(
    ds=ds_nairobi,
    lat=lat_nairobi,
    lon=lon_nairobi,
    data_path=data_path,
    grid=grid,
    ml_vars=ml_vars,
    B_vars=B_vars,
)

In [ ]:
path = '/scratch/projects/phase01/rab/B06/PROCESSED_ERA5/Data/'

era5_kuala = xr.open_zarr(path + 'kuala_era5.zarr')
era5_nairobi = xr.open_zarr(path  +'nairobi_era5.zarr')
era5_sancrist = xr.open_zarr(path  +'sancrist_era5.zarr')

In [ ]:
bins_kuala = np.arange(0, 25, 0.1)
bins_nairobi = np.arange(0, 25, 0.1)
bins_sancrist = np.arange(0, 25, 0.1)

print('bins ready')

def bin_profiles(ds, bins):
    profiles = []

    bin_width = bins[1] - bins[0]
    bin_centers = bins[:-1] + bin_width / 2

    for day in ds.day.values:
        p = ds.sel(day=day)

        p = p.where(
            p["GeopAlt"].notnull(),
            drop=True
        )

        binned = p.groupby_bins(
            "GeopAlt",
            bins=bins,
            labels=bin_centers,
            right=False,
        ).mean(skipna=True)

        binned = (
            binned
            .rename({"GeopAlt_bins": "height"})
            .assign_coords(height=bin_centers)
            .expand_dims(day=[day])
        )

        profiles.append(binned)

    return xr.concat(profiles, dim="day")

ds_kuala = bin_profiles(data_kuala, bins_kuala)
print('done')
ds_nairobi = bin_profiles(data_nairobi, bins_nairobi)
ds_sancrist = bin_profiles(data_sancrist, bins_sancrist)
#ds_quito = bin_profiles(data_quito, bins_quito)
ds_kuala = calc_lrt(ds_kuala)
ds_nairobi = calc_lrt(ds_nairobi)
ds_sancrist = calc_lrt(ds_sancrist)

In [ ]:
threshold = 16

print('Kuala')
print(len(ds_kuala.day))
print((ds_kuala.z_lrt.values < threshold).sum())
print((ds_kuala.z_lrt.values < threshold).sum()/len(ds_kuala.day)*100)

print('Nairobi')
print(len(ds_nairobi.day))
print((ds_nairobi.z_lrt.values < threshold).sum())
print((ds_nairobi.z_lrt.values < threshold).sum()/len(ds_nairobi.day)*100)

print('Quito')
print(len(ds_sancrist.day))
print((ds_sancrist.z_lrt.values < threshold).sum())
print((ds_sancrist.z_lrt.values < threshold).sum()/len(ds_sancrist.day)*100)

In [ ]:
threshold = 16000

print('Kuala')
print(len(era5_kuala.time))
print((era5_kuala.z_lrt.values < threshold).sum())
print((era5_kuala.z_lrt.values < threshold).sum()/len(era5_kuala.time)*100)

print('Nairobi')
print(len(era5_nairobi.time))
print((era5_nairobi.z_lrt.values < threshold).sum())
print((era5_nairobi.z_lrt.values < threshold).sum()/len(era5_nairobi.time)*100)

print('Quito')
print(len(era5_sancrist.time))
print((era5_sancrist.z_lrt.values < threshold).sum())
print((era5_sancrist.z_lrt.values < threshold).sum()/len(era5_sancrist.time)*100)

In [ ]:
ffig, ax = plt.subplots(figsize=(7, 5))

threshold = 16

ax.hist(
    ds_kuala.z_lrt.values,
    bins=100,
    density=True,
    histtype="step",
    color="#46179E",
    linewidth=2,
    label="Kuala Lumpur",
)

ax.hist(
    ds_nairobi.z_lrt.values,
    bins=100,
    density=True,
    histtype="step",
    color="#A05437",
    linewidth=2,
    label="Nairobi",
)

ax.hist(
    ds_sancrist.z_lrt.values,
    bins=100,
    density=True,
    histtype="step",
    color="#BEAB3F",
    linewidth=2,
    label="Quito",
)

ax.axvline(
    threshold,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Threshold = {threshold:.2f} km",
)

ax.set_xlabel("Lapse-rate tropopause height [km]")
ax.set_ylabel("Relative frequency")

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.8,
    alpha=0.6,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False)

ax.set_xlim(13,20)
plt.show()

In [ ]:
p0 = 1000.0
Rd = 287.0
cp = 1004.0
g = 9.81

def add_stability_variables_shadoz(ds):

    angle = np.deg2rad(ds.Wind_Dir)

    u = -ds.Wind_Spd * np.sin(angle)
    v = -ds.Wind_Spd * np.cos(angle)

    du_dz = u.differentiate("height") / 1000
    dv_dz = v.differentiate("height") / 1000

    shear = du_dz**2 + dv_dz**2

    T = ds.Temp + 273.15
    P = ds.Press

    theta = T * (p0 / P) ** (Rd / cp)

    dtheta_dz = theta.differentiate("height") / 1000

    N2 = (g / theta) * dtheta_dz

    Ri = N2 / shear
    return ds.assign(
        shear=shear,
        N2=N2,
        Ri=Ri,
    )

ds_kuala = add_stability_variables_shadoz(ds_kuala)
ds_nairobi = add_stability_variables_shadoz(ds_nairobi)
ds_sancrist = add_stability_variables_shadoz(ds_sancrist)

In [ ]:
def o3_to_ppbv(o3):
    M_air = 28.9644   # g/mol
    M_o3 = 48.00      # g/mol

    return o3 * (M_air / M_o3) * 1e9

def specific_humidity_to_ppmv(q):
    M_h2o = 18.01528  # g/mol
    M_dry = 28.9644   # g/mol

    x_h2o = (q / M_h2o) / (
        q / M_h2o + (1 - q) / M_dry
    )

    return x_h2o * 1e6

def add_stability_variables(ds):

    du_dz = ds.u.differentiate("level")/ ds.height.differentiate("level")
    dv_dz = ds.v.differentiate("level")/ ds.height.differentiate("level")

    shear = du_dz**2 + dv_dz**2

    Ri = ds.N2.where(shear > 1e-8) / shear.where(shear > 1e-8)

    return ds.assign(
        shear=shear,
        Ri=Ri,
    )

era5_kuala = add_stability_variables(era5_kuala)
era5_nairobi = add_stability_variables(era5_nairobi)
era5_sancrist = add_stability_variables(era5_sancrist)

In [ ]:
ds_kuala_inversion = ds_kuala.where(ds_kuala.z_lrt <= 16, drop = True)
ds_kuala_rest = ds_kuala.where(ds_kuala.z_lrt > 16, drop = True)
era5_kuala_inversion = era5_kuala.where(era5_kuala.z_lrt.compute() <= 16000, drop = True)
era5_kuala_rest = era5_kuala.where(era5_kuala.z_lrt.compute() > 16000, drop = True)

# height_above_lrt = ((ds_kuala.height >= ds_kuala.z_lrt)& (ds_kuala.height <= ds_kuala.z_lrt + 2))
# ds_kuala_inversion = ds_kuala.where(((ds_kuala.Ri < 1) & height_above_lrt).any(dim="height"),drop=True,)
# ds_kuala_rest = ds_kuala.drop_sel(day=ds_kuala_inversion.day)

# height_above_lrt_era5 = ((era5_kuala.height >= era5_kuala.z_lrt)& (era5_kuala.height <= era5_kuala.z_lrt + 2000))
# era5_kuala_inversion = era5_kuala.where(((era5_kuala.Ri.compute() < 1) & height_above_lrt_era5.compute()).any(dim="level"),drop=True,)
# era5_kuala_rest = era5_kuala.drop_sel(time=era5_kuala_inversion.time)

g = 9.81
Rd = 287.05
cp = 1004.0
p0 = 1000.0  # hPa


def mean_profile(ds, var):
    """Mean vertical profile over day."""
    return ds[var].mean(dim="day")

def mean_profile_Era5(ds, var):
    """Mean vertical profile over day."""
    return ds[var].mean(dim="time")

def O3_Gradient(ds):
   do3dz = ds.O3_ppmv.differentiate(coord = 'height')*1000

   return do3dz.mean(dim = 'day')

def O3_Gradient_Era5(ds):
    o3_ppbv = o3_to_ppbv(ds.o3)

    do3_dlevel = o3_ppbv.differentiate(coord="level")
    dz_dlevel = ds["height"].differentiate(coord="level")

    do3_dz = do3_dlevel / dz_dlevel

    # Convert from ppbv/m to ppbv/km if height is in metres
    do3_dz = do3_dz * 1000

    return do3_dz.mean(dim="time")

datasets_1 = {
    "Kuala Lumpur": ds_kuala,
    "Nairobi": ds_nairobi,
    "San Cristobal": ds_sancrist,
    "Kuala Lumpur (inversion)": ds_kuala_inversion,
    "Kuala Lumpur (no inversion)": ds_kuala_rest,
}

datasets_2 = {
    "Kuala Lumpur": era5_kuala,
    "Nairobi": era5_nairobi,
    "San Cristobal": era5_sancrist,
    "Kuala Lumpur (inversion)": era5_kuala_inversion,
    "Kuala Lumpur (no inversion)": era5_kuala_rest,
}

colors = {
    "Kuala Lumpur": "#46179E",
    "Nairobi": "#A05437",
    "San Cristobal": "#BEAB3F",
    "Kuala Lumpur (inversion)": "#46179E",
    "Kuala Lumpur (no inversion)": "#46179E",
}

linestyles = {
    "Kuala Lumpur": "-",
    "Nairobi": "-",
    "San Cristobal": "-",
    "Kuala Lumpur (inversion)": ":",
    "Kuala Lumpur (no inversion)": "-.",
}

profiles_1 = {}

for name, ds in datasets_1.items():

    profiles_1[name] = {
        "height": ds.height * 1000,
        "trop_rel_z": mean_profile(ds, "trop_rel_z") * 1000,
        "O3": mean_profile(ds, "O3_ppmv") * 1000,
        "dO3dz" : O3_Gradient(ds),
        "shear": mean_profile(ds, "shear"),
        "Temp": mean_profile(ds, "Temp"),
        "N2": mean_profile(ds, "N2"),
        "Ri": mean_profile(ds, "Ri"),
        "tropopause": ds.z_lrt.mean(dim="day") * 1000,
    }

profiles_2 = {}

for name, ds in datasets_2.items():

    profiles_2[name] = {
        "height": ds.height.mean(dim = 'time'),
        "trop_rel_z": mean_profile_Era5(ds, "trop_rel_z") * 1000,
        "O3": o3_to_ppbv(ds.o3).mean(dim = 'time'),
        "dO3dz" : O3_Gradient_Era5(ds),
        "shear": mean_profile_Era5(ds, "shear"),
        "Temp": mean_profile_Era5(ds, "t"),
        "N2": mean_profile_Era5(ds, "N2"),
        "Ri": mean_profile_Era5(ds, "Ri"),
        "tropopause": ds.z_lrt.mean(dim="time"),
    }

titles = [
    "Temperature",
    "Richardson Number",
    "Ozone Gradient",
    "Ozone",
]

panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]

xdata_1 = {
    "Temperature": {name: p["Temp"] + 273.15 for name, p in profiles_1.items()},
    "Richardson Number": {name: p["Ri"] for name, p in profiles_1.items()},
    "Ozone Gradient": {name: p["dO3dz"] for name, p in profiles_1.items()},
    "Ozone": {name: p["O3"] for name, p in profiles_1.items()},
}

xdata_2 = {
    "Temperature": {name: p["Temp"] for name, p in profiles_2.items()},
    "Richardson Number": {name: p["Ri"] for name, p in profiles_2.items()},
    "Ozone Gradient": {name: p["dO3dz"] for name, p in profiles_2.items()},
    "Ozone": {name: p["O3"] for name, p in profiles_2.items()},
}

xlabels = {
    "Temperature": "Temperature [K]",
    "Richardson Number": "Richardson number",
    "Ozone Gradient": r"$dO_3/dz$ [ppbv km$^{-1}$]",
    "Ozone": "Ozone [ppbv]",
}

fig, axes = plt.subplots(
    2, 4,
    figsize=(18, 9),
    sharey=True,
)

# Columns 0-1: SHADOZ; columns 2-3: ERA5
plot_specs = [
    (0, 0, "Temperature", profiles_1, xdata_1, "a)"),
    (0, 1, "Richardson Number", profiles_1, xdata_1, "b)"),
    (1, 0, "Ozone Gradient", profiles_1, xdata_1, "c)"),
    (1, 1, "Ozone", profiles_1, xdata_1, "d)"),
    (0, 2, "Temperature", profiles_2, xdata_2, "e)"),
    (0, 3, "Richardson Number", profiles_2, xdata_2, "f)"),
    (1, 2, "Ozone Gradient", profiles_2, xdata_2, "g)"),
    (1, 3, "Ozone", profiles_2, xdata_2, "h)"),
]

for row, col, title, profiles, xdata, panel in plot_specs:
    ax = axes[row, col]

    for name, x in xdata[title].items():
        linestyle = linestyles[name]

        ax.plot(
            x,
            profiles[name]["height"],
            color=colors[name],
            linestyle=linestyle,
            linewidth=3 if linestyle == "-" else 2.2,
            label=name,
        )

        ax.axhline(
            profiles[name]["tropopause"],
            color=colors[name],
            linestyle=linestyle,
            linewidth=1.3,
            alpha=0.6,
        )

    ax.set_title(title, fontsize=13, pad=10)
    ax.text(
        0.03, 0.97, panel,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )

    ax.set_xlabel(xlabels[title], fontsize=11)
    ax.set_ylim(14000, 19000)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.6)
    ax.tick_params(axis="both", labelsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Group headings
fig.text(0.25, 0.96, "SHADOZ", ha="center", fontsize=15, fontweight="bold")
fig.text(0.75, 0.96, "ERA5", ha="center", fontsize=15, fontweight="bold")

fig.add_artist(plt.Line2D(
    [0.52, 0.52], [0.1, 1],
    transform=fig.transFigure,
    color="grey",
    linewidth=2,
    linestyle="--",
))

# Reference line for Richardson number
axes[0, 1].axvline(
    1, color="black", linestyle="--", linewidth=1.2, alpha=0.7
)
axes[0, 3].axvline(
    1, color="black", linestyle="--", linewidth=1.2, alpha=0.7
)

# Axis limits
axes[0, 0].set_xlim(190, 230)
axes[0, 2].set_xlim(190, 230)

axes[0, 1].set_xlim(.5, 10)
axes[0, 3].set_xlim(.5, 10)
axes[0, 1].set_xscale('log')
axes[0, 3].set_xscale('log')

axes[1, 0].set_xlim(0, 300)
axes[1, 2].set_xlim(0, 300)

axes[1, 1].set_xlim(0, 400)
axes[1, 3].set_xlim(0, 400)

axes[0, 0].set_ylabel("Height [m]", fontsize=11)
axes[1, 0].set_ylabel("Height [m]", fontsize=11)

# Shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=5,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
)

fig.subplots_adjust(
    left=0.07,
    right=0.97,
    top=0.89,
    bottom=0.13,
    wspace=0.10,
    hspace=0.30,
)

# plt.savefig(
#     "../../Plots/PaperReady/Shadoz.png",
#     format="PNG",
#     dpi=300,
#     bbox_inches="tight",
# )
